# LP Duality and the Diet Problem
Based on chapter 2 of the AMPL book and section 6.2 of *Decision Making, Models and Algorithms: A First Course* by Saul I. Gass.

**Objectives**
- Introduce a historically important linear program
- Practice taking the dual of a minimization problem
- Think about solving linear programs and tweaking solutions
- Demonstrate the idea of sensitivity analysis in linear programming

**Reading:** This lab uses a diet problem instance provided by NEOS. Please read the following description of the diet problem and its history: https://neos-guide.org/case-studies/om/the-diet-problem/. Their online diet problem solver can be found on the same page.

**Textbook Reading:** Chapter 13, Linear Programming Duality

**Brief description:** In this lab, we will consider one of the most famous (and one of the earliest) applications of linear programming — the diet problem.

 <br>


In [ ]:
# imports -- don't forget to run this cell
import pandas as pd
from ortools.linear_solver import pywraplp as OR

**Names of group members:**

**Who will be "driving" for the first portion of this lab: (write their name and NetID here)**

_As a reminder, only the person driving is supposed to touch the keyboard. Everyone in the group must read the lab, help answer questions, etc. Also, the driver must still discuss/explain/etc question answers with their group—they cannot just write down answers without discussion even if they know them._

## Part 1: The Diet Problem

Suppose that you have a choice of two cereals for breakfast: Krunchies (K) or Crispies (C). Breakfast is the
most important meal of the day, so you want to get sufficient Thiamine, Niacin, and caloric intake for breakfast. Being a college student, you want to do so as cheaply as possible – taste gets thrown into the wind! Price and nutritional info for these two cereals is summarized in the two tables below in **per ounce of cereal**.

In [ ]:
cereals = pd.read_csv('data/diet_cereal/cereal_foods.csv', index_col=0)
cereal_nutrients = pd.read_csv('data/diet_cereal/cereal_nutrients.csv', index_col=0)
display(cereals)
display(cereal_nutrients)

**Q1.1:** Suppose you just ate Krunchies. How many ounces of Krunchies would you need to eat to satisfy the
three nutritional requirements? How much would this cost?

**A:**

_Write your answer here._

**Q1.2:** Suppose you just ate Crispies. How many ounces of Crispies would you need to eat to satisfy the three
nutritional requirements? How much would this cost?

**A:**

_Write your answer here._

**Q1.3:** Now let’s write an optimization problem to model this problem. Let $K$ be a decision variable for the amount of Krunchies you eat and $C$ be a decision variable for the amount of Crispies you eat. Write an objective function, encoding that you minimize total cost (as a function of $K$ and $C$).

**A:**

_Write your answer here._

**Q1.4:** Write three constraints: one enforcing that you get at least 1 mg of Thiamine, one enforcing that you
get at least 5 mg of Niacin, and one enforcing that you get at least 400 calories. Also write constraints
that $K$ and $C$ are nonnegative.

**A:**

_Write your answer here._

**Q1.5:** Implement this model in OR-Tools. The basic structure has been given to you.

**A:** _Don't write anything here, modify the code block below!_

#### OR-Tools example (syntax only)

For an unrelated model, suppose `choices` is a list of options, `x` stores the decision variable for each option, and `cost` and `output` store the corresponding coefficients. Here are examples of how you could write objectives and constraints for OR-Tools.

```python
# Objective
m.Minimize(sum(cost[i] * x[i] for i in choices))

# Constraint
m.Add(sum(output[i] * x[i] for i in choices) >= required_output)
```

Use this pattern with the variables and coefficients from the cereal tables above. Each `m.Add(...)` call adds one constraint.


In [ ]:
def small_diet():
    """A linear program for solving a diet problem."""
    # define the model
    m = OR.Solver('cereal_diet', OR.Solver.CLP_LINEAR_PROGRAMMING);

    # TODO: Define the decision variables.
    # name = m.NumVar(LOWER_BOUND, UPPER_BOUND, "VARIABLE_NAME")
    # If you want to use infinity as a bound, you can use m.infinity()


    # objective function
    # TODO: Define the objective function.
    # m.Minimize(OBJ_FUNCTION) or m.Maximize(OBJ_FUNCTION)


    # constraints
    # TODO: Define the constraints.
    # m.Add(CONSTRAINT_EQUATION)

    
    
    return m, m.variables()

In [ ]:
def solve(m):
    m.Solve()
    print('Solution:')
    print('Objective value =', m.Objective().Value())
    for var in m.variables():
        print(var.name(), ':',  var.solution_value())

In [ ]:
m,x = small_diet()
solve(m)

**Q1.6:** Run the cell above. What was the optimal solution and its objective value?

**A:**

_Write your answer here._

## Part 2: Diet Problem Duality

In the Linear Programming Duality handout, we found an upper bound to a maximization problem. In this case, we have a minimization problem, so instead we will try to get a lower bound on the solution.

For convenience, here is the linear program we formulated in the previous section.
\begin{align*}
\min \quad & 3.8K + 4.2C \\
\text{s.t.} \quad &  0.1K + 0.25C \geq 1 & (1)\\
\quad &  1K + 0.25C \geq 5 & (2)\\
\quad & 110K + 120C \geq 400 & (3)\\
\quad & K,C \geq 0
\end{align*}



**Q2.1:** Look at constraint (1). Why is $1$ a lower bound on the optimal solution?

**A:**

_Write your answer here._

**Q2.2:** What abound constraint (2)? What lower bound can you get directly from constraint two?

**A:**

_Write your answer here._

**Q2.3:** We can also add together constraints to get more inequalities that must be true. What is the best lower bound you can get by adding together two constraints?

**A:**

_Write your answer here._

**Q2.4:** Remember that if you multiply an inequality by a nonnegative value, the resulting inequality is still a true statement. Find a lower bound by multiplying (3) by some value.

**A:**

_Write your answer here._

**Q2.5:** Combining positive multiples of existing constraints gives us more valid constraints. Let's look at some: 

$3 \times (constraint (1)) + 2 \times (constraint (2)) + 1/50 \times (constraint (3))$

Does this new inequality give us a lower bound on the optimal value? Why or why not?

**A:**

_Write your answer here._

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

Let $y_1$, $y_2$, and $y_3$ be the coefficients that we multiply constraints (1), (2), and (3) by respectively.

**Q2.6:** What happens if we choose $y_1 < 0$? Why can we not use this to find a lower bound?

**A:**

_Write your answer here._

You should've seen that you need constraints on the sign of $y_1$, which means we also need constraints on the sign of $y_2$ and $y_3$. (Think for a second--what should those look like?)

**Q2.7:** What else must be true for these constraints to be a lower bound on the optimal value? You should write two inequalities here. (Hint: think about Q2.3 and Q2.4)

**A:**

_Write your answer here._

**Q2.8:** What is the value of the lower bound that we get (in terms of $y_1, y_2, y_3$)?

**A:**

_Write your answer here._

We've just taken the dual of our minimization problem! Maximizing this value subject to the new constraints we wrote on $y_1, y_2, y_3$ will give us the best possible lower bound we can find using this method.

Implement the dual in OR-tools. The basic structure has been given to you. 

In [ ]:
def small_diet_dual():
    """A linear program for solving a diet problem."""
    # define the model
    m = OR.Solver('cereal_diet', OR.Solver.CLP_LINEAR_PROGRAMMING);

    # TODO: Define the decision variables.
    # name = m.NumVar(LOWER_BOUND, UPPER_BOUND, "VARIABLE_NAME")
    # If you want to use infinity as a bound, you can use m.infinity()

    

    # objective function
    # TODO: Define the objective function.
    # m.Minimize(OBJ_FUNCTION) or m.Maximize(OBJ_FUNCTION)

    

    # constraints
    # TODO: Define the constraints.
    # m.Add(CONSTRAINT_EQUATION)

    
    
    return m, m.variables()

In [ ]:
d,x = small_diet_dual()
solve(d)

**Q2.9:** What is the value of the lower bound that we've found? How can you use this to prove that the solution we found to the diet problem is optimal?

**A:**

_Write your answer here._

**Q2.10:** Suppose you increase the requirement for thiamine to 2mg. How much does your optimal value change by? Edit the model below and run the cells to find the new solution.

**A:**

_Write your answer here._

In [ ]:
def small_diet():
    """A linear program for solving a diet problem."""
    # define the model
    m = OR.Solver('cereal_diet', OR.Solver.CLP_LINEAR_PROGRAMMING);

    K = m.NumVar(0, m.infinity(), 'K')
    C = m.NumVar(0, m.infinity(), 'C')
    
    m.Minimize(3.8*K + 4.2*C)

    # TODO: Update thiamine requirement
    m.Add(0.1*K + 0.25*C >= 1)
    m.Add(1*K + 0.25*C >= 5)
    m.Add(110*K + 120*C >= 400)
    
    return m, m.variables()

In [ ]:
m,x = small_diet()
solve(m)

**Q2.11:** Now suppose you set the thiamine requirement back to 1mg, but change the niacin requirement to 6mg. How much does your optimal value change by compared to the original solution? Edit the same model above and run the cells to find the new solution.

**A:**

_Write your answer here._

**Q2.12:** How do the changes in the optimal value relate to the dual values we found above?

**A:**

_Write your answer here._

The dual value of a constraint tells us how much the optimal objective value would change if we relaxed or tightened that constraint by one unit. In other words, it measures how "valuable" an additional unit of a limited resource is. Constraints with larger-magnitude dual values have a greater impact on the optimal solution, while constraints with a dual value of zero are not currently limiting the optimum. However, these interpretations are only accurate for small changes to the problem data. Nonetheless, this makes dual values a very useful tool for understanding which resources are most valuable and for predicting the effect of small changes to the model.

## Part 3: Generalizing the Diet Problem

We now know that if we expect to live a long and healthy life, we must have a well-balanced diet. Too much fat or sodium in our diet can lead to serious health problems. Similarly, diets deficient in essential vitamins and minerals should be avoided. 

This part of the lab is aimed at formulating this problem as a linear programming problem. We can view the diet problem in the following way. We are given a variety of foods that we could buy to achieve a balanced diet. For example, we might consider a diet consisting of 2% milk, spaghetti (with sauce), peanut butter, wheat bread, tomato soup, and bagels. To make things simpler, we will specify the variables of this linear programming formulation. We will use $x$_$\textit{(food-type)}$ to specify the number of daily servings of food-type that you are willing to consume. First of all, we wish to write constraints that capture whether one can satisfy certain daily requirements with just these foods.

**Q3.1:** Write constraints to ensure we eat at most 10 servings/day of each food-type (We can call this the boredom constraint).

**A:**

_Write your answer here._

Next, consider the total number of calories consumed. A proper diet requires that you consume between 2000 and 2250 calories per day. 

**Q3.2:** Write two constraints to ensure we eat an appropriate number of total calories. To write this constraint you need to know how many calories are in one serving of each of the food-types. Let $a_{i}$ be the number of calories in one serving of food $i$.

**A:**

_Write your answer here._

Of course, we could repeat this sort of constraint with any number of nutritional requirement, such as cholesterol, fat, sodium, dietary fiber, carbohydrates, protein, vitamin A, vitamin C, calcium, and iron. In general, let $a_{ij}$ be the amount of nutrient $j$ in one serving of food $i$. Then our constraints take the form
$$(\text{min amount of nutrient } j) \leq \sum_i a_{ij} x_i \leq (\text{max amount of nutrient } j), \quad \text{for all nutrients } j$$

**Q3.3:** The objective function is to minimize the cost of each day’s diet. Express this objective function in terms of the decision variables, given that $c_i$ is the cost of one serving of food $i$.

**A:**

_Write your answer here._

**Q3.4:** What does it mean if your linear program for this diet problem is infeasible?

**A:**

_Write your answer here._

**Q3.5:** How could you change the input data or optimization model to try to correct this?
    
**A:**

_Write your answer here._

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

Let's use OR-Tools to implement this generalized diet problem model!

**Q3.6:** Complete the model below by defining the objective function.

In [ ]:
def diet(foods, nutrients, integer=False):
    """A model for solving the diet problem.
    
    Args:
        foods (pd.DataFrame): Foods with cost per serving, min and max servings, and nutrients per serving.
        nutrients (pd.DataFrame): Nutrients with min and max bounds.
    """
    FOODS = list(foods.index)                                 # foods
    NUTRIENTS = list(nutrients.index)                         # nutrients
    c = foods['Cost'].to_dict()                               # cost per serving of food 
    f_min = foods['Min'].to_dict()                            # lower bound of food serving
    f_max = foods['Max'].to_dict()                            # upper bound of food serving
    n_min = nutrients['Min'].to_dict()                        # lower bound of nutrient
    n_max = nutrients['Max'].to_dict()                        # upper bound of nutrient  
    a = foods[list(nutrients.index)].transpose().to_dict()    # amt of nutrients per serving of food
    
    # define model
    if integer:
        m = OR.Solver('diet', OR.Solver.CBC_MIXED_INTEGER_PROGRAMMING)
    else:
        m = OR.Solver('diet', OR.Solver.GLOP_LINEAR_PROGRAMMING)
        
    # decision variables
    x = {}    
    for i in FOODS:
        if integer:
            x[i] = m.IntVar(0, m.infinity(), 'x_%s' % (i)) 
        else:
            x[i] = m.NumVar(0, m.infinity(), 'x_%s' % (i)) 
        
    # objective function.
    # TODO: Define the objective function.
    # m.Minimize() or m.Maximize()

    
    
    # enforce lower and upper bound on food servings
    for i in FOODS:
        m.Add(x[i] >= f_min[i], name='lb_%s' % (i))
        m.Add(x[i] <= f_max[i], name='ub_%s' % (i))
    
    # enforce lower and upper bound on nutrients 
    for j in NUTRIENTS:
        m.Add(sum(a[i][j]*x[i] for i in FOODS) >= n_min[j], name='lb_%s' % (j))
        m.Add(sum(a[i][j]*x[i] for i in FOODS) <= n_max[j], name='ub_%s' % (j))
        
    return m, x

In [ ]:
def solve(m):
    '''Used to solve a model m.'''
    if m.Solve() == m.INFEASIBLE:
        print('infeasible.')
    else:
        print('Objective = %f \n' % (m.Objective().Value()))  
        x = {var.name() : var.solution_value() for var in m.variables()}
        for i in x:
            if x[i] != 0:
                print('%s: %s' % (i, x[i]))
                
def dual_values(m):
    '''Return the non-zero dual values.'''
    for i in m.constraints():
        name = i.name()
        dual_value = i.dual_value()
        if dual_value != 0:
            print(i.name(), i.dual_value())
            
def nutrient_values(x, m, foods, nutrients):
    '''Return the value of each nutrient in this solution.'''
    FOODS = list(foods.index)                                 
    NUTRIENTS = list(nutrients.index)                         
    a = foods[list(nutrients.index)].transpose().to_dict()    
    for j in NUTRIENTS:
        value = sum(a[i][j]*x[i].solution_value() for i in FOODS)
        print('%s: %f' % (j, value))

To make sure our model is correct, let's use it to solve the cereal input from **Part 1**. In order to use this new model, we will have to slightly modify our input. Recall the cereal input:

In [ ]:
display(cereals)
display(cereal_nutrients)

**Q3.7:** We need to specify an "Max" amount (upper bound) for each nutrient and "Min" and "Max" (lower and upper bounds) for each food. What should these be? (don't forget about the boredom constraint!)

**A:**
_Add your answers below._
* Min for Krunchies: 
* Min for Crispies: 
* Max for Krunchies: 
* Max for Crispies: 
* Max for Thiamine:  
* Max for Niacin: 
* Max for Calories: 

**Q3.8:** Since these values are the same for all nutrients and for all the cereals, we simplified things for you--you only need to set everything once for the cereals and nutrients. Add your answer from Q3.7 in the code cell below.

In [ ]:
# TODO: Set the values below.
# Hint: use float('inf') for infinity.

# cereals['Min'] =  
# cereals['Max'] = 
# cereal_nutrients['Max'] = 



In [ ]:
m,x = diet(cereals, cereal_nutrients)
solve(m)

**Q3.9:** Run the cell above to solve the model. Did you get the same solution as you did in **Part 1**? Are any of the min/max serving constraints tight?

**A:**

_Write your answer here._

## Part 4: Solving the Diet Problem

Let's start by solving the diet problem input given in the beginning of **Part 2**. It uses a subset of all the foods in the NEOS dataset. Try to solve the model. You should see it is infeasible.

In [ ]:
foods = pd.read_csv('data/diet_neos/neos_foods.csv', index_col=0)
sm_foods = foods.loc[['2% Lowfat Milk', 'Spaghetti W/ Sauce', 
                      'Peanut Butter', 'Wheat Bread', 'Tomato Soup', 'Bagels']]
nutrients = pd.read_csv('data/diet_neos/neos_nutrients.csv', index_col=0)

In [ ]:
display(sm_foods.head())
display(nutrients)

In [ ]:
m,x = diet(sm_foods, nutrients)
solve(m)

**Q4.1:** Take a look at the specifics of the nutrient requirements and the nutrient contents, and try to understand why there is no feasible solution. Which nutrient(s) do you think were (most) responsible for the infeasiblity? Your answer doesn't need to be precise, but try to give some reasoning and intuition for why you suggested those nutrients.

**A:**

_Write your answer here._

In [ ]:
m,x = diet(foods, nutrients)
solve(m)

**Q4.2:** Now consider a much wider set of food-types (this diet problem is solved above). What is the optimal solution and objective value? (Hint: the solve function does not print foods with 0 servings in the solution.)

**A:**

_Write your answer here._

**Q4.3:** How many different food-types are included in this diet (There were 62 potential foods)? Is this surprising to you?

**A:**

_Write your answer here._

In [ ]:
nutrient_values(x, m, foods, nutrients)

**Q4.4:** The function above returns how much of each nutrient you will actually consume. For which nutrients are you at your upper limit? Such a constraint is said to be tight.

**A:**

_Write your answer here._

**Q4.5:** For which are you at your lower limit? (That is, which lower limits are tight?)

**A:**

_Write your answer here._

**Q4.6:** Are any of the minimum and maximum number of serving constraints tight?

**A:**

_Write your answer here._

**At this point, please switch drivers. Take a minute to make sure everyone understands this section and is on the same page.**

**Who will be "driving" for the next portion of this lab: (write their name and NetID here)**

You should have included carbohydrates in the list of nutrients that is at its upper limit in your diet. Use the function below to get the dual values of this solution. (Hint: this function only shows non-zero dual values.)

In [ ]:
dual_values(m)

**Q4.7:** Suppose you increase the allowance for carbohydrates to 301 grams. Run the code cell below to see how the optimal diet changes. What is the new optimal solution, and by how much does its cost change? How does this relate to the dual value for the constraint that upper bounds carbohydrates?

**A:**

_Write your answer here._

In [ ]:
tmp = nutrients.copy()
tmp.at['Carbohydrates', 'Max'] = 301
m,x = diet(foods, tmp)
solve(m)

Return to the dual information. The dual value for the carbohydrates upper bound constraints is given there. This information can be used in the following way. Suppose that a dual value for a particular nutrient's upper bound constraint is $−c$ , (i.e., it is negative). Now suppose that we may have one more unit of this nutrient in our diet, and when we find the optimal solution for this modified data, the foods that make up the optimal diet are unchanged.

**Q4.8:** How does the cost of the optimal solution to the new data compare to the optimal solution to the original data?    

**A:**

_Write your answer here._

**Q4.9:** In fact, the dual cost tells how much cheaper the new solution will be: it will be $c$ units cheaper. If we were to increase the max on total fat by 1 (making it 66), what would you expect the new cost to be? Solve the model to see if it was as expected.

**A:**

_Write your answer here._

In [ ]:
tmp = nutrients.copy()
tmp.at['Total_Fat', 'Max'] = 66
m,x = diet(foods, tmp)
solve(m)

**Q4.10:** Now allow up to 320 grams of carbohydrates in your diet. How does this change things? Does the analogous calculation to the one that you just did still work? Why do you think that this is?

**A:**

_Write your answer here._

In [ ]:
tmp = nutrients.copy()
tmp.at['Carbohydrates', 'Max'] = 320
m,x = diet(foods, tmp)
solve(m)

**Q4.11:** Go back to the default data, its optimal solution, and the dual and constraint information for it. Which constraint do you think it would be most profitable to violate by one unit? Can you give an intuitive explanation of this?
    
**A:**

_Write your answer here._

**Q4.12:** As you can tell from the output, the optimal solution largely calls for fractional numbers of servings of the various food-types. Why might this be unrealistic in practice? (Hint: how do we buy food?) When might fractional solutions be justified? (Hint: could we interpret the solution to the optimization problem as something other than daily nutrition guidance? Are there specific food types that fractional solutions make more sense for?)

**A:**

_Write your answer here._

**Q4.13:** Suppose that you are buying just one day’s worth of food, and you must buy an integral number of servings of each food-type. Can you still use this output to construct a diet? Is this new selection an optimal diet subject to the restriction that the number of servings bought be integral?

**A:**

_Write your answer here._

**Q4.14:** Without worrying about how to compute the optimal integer solution, how would you expect the cost of the optimal integer solution compare to the cost of our linear program’s optimal solution? 

**A:**

_Write your answer here._

**Q4.15:** Run the cell below to solve the model with the restriction that servings must be integer. What is the optimal solution and objective value? How does it compare to the linear relaxation?

**A:**

_Write your answer here._

In [ ]:
m,x = diet(foods, nutrients, integer=True)
solve(m)

**Q4.16:** What do you think about eating the diet proposed by the output? How might you modify the constraints to get something more to your liking?

**A:**

_Write your answer here._

### Key Takeaways

Congratulations! In this lab, you’ve built and analyzed linear programming models for the diet problem. You should have learned about:

* Formulate a linear program by identifying the decision variables, objective function, and constraints.
* Model real-world problems using lower and upper bounds, constraints (such as nutritional requirements), and objective functions (such as ingredient costs).
* Interpret dual values (shadow prices) as the value of relaxing a constraint and use them to understand which constraints are most limiting.
* Recognize the limitations of dual values, including that they provide accurate sensitivity information only for sufficiently small changes to the model.
* Understand that conflicting constraints may make it impossible to satisfy all requirements simultaneously, and lead to infeasible models.
* Write linear programs using abstract variables for coefficients (such as $a_{ij}$ for the amount of nutrient $i$ contained in one serving of food $j$) and sum notation that works for arbitrary sets (for the diet problem, of foods and nutrients) instead of a single hard-coded instance.
* Understand the strengths and limitations of linear programming, including why LPs often produce fractional solutions and when those solutions may or may not be reasonable in practice.

The diet problem is a classic example of how optimization can be used to make the best decision subject to many competing constraints. The same modeling techniques can be used for scheduling, finance, supply chain management, energy systems, allocation for emergency healthcare services, and many more applications.